# 🎬 Movie Recommendation System

This notebook presents the development of a movie recommendation system for the Minor Data-Driven Decision Making individual assignment.

The goal of this project is to build a minimum viable data product (MVP) using real-world movie data. The final product recommends movies based on genre and minimum rating.

This notebook includes:
- Project background
- Learning inspiration from selected videos
- Dataset explanation
- Data cleaning
- Genre processing
- Recommendation logic
- Example output
- Reflection and insights


## 1. Learning Inspiration

This project was inspired by two selected videos:

- **AI Python for Beginners**
- **Become a Data Storyteller with Streamlit**

I selected these videos because they focus on practical implementation rather than only theory.

The first video helped me understand how Python can be used to build simple AI and data-driven applications.  
The second video showed how Streamlit can be used to turn data into an interactive and user-friendly experience.

These videos inspired me to build a simple movie recommendation system that is easy to use and easy to understand.


## 2. Import Libraries

In this step, I import the Python libraries required for the project.

- `pandas` is used for data loading, cleaning, and filtering.
- `ast` is used to convert genre information from text format into Python lists.


In [2]:
import pandas as pd
import ast

## 3. Load the Dataset

The dataset used in this project is `movies_metadata.csv`.

This file contains real-world movie metadata such as:
- Movie title
- Overview
- Genres
- Average rating
- Release date

The dataset is not included in the GitHub repository because it is too large. It should be downloaded separately and placed in the same folder as this notebook.


In [3]:
movies = pd.read_csv("movies_metadata.csv", low_memory=False)

movies.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


## 4. Select Relevant Columns

The original dataset contains many columns that are not required for this MVP.

For this project, I only use the columns that are relevant to the recommendation system:
- `title`
- `overview`
- `genres`
- `vote_average`
- `release_date`

This keeps the project focused and easier to manage.


In [4]:
movies = movies[["title", "overview", "genres", "vote_average", "release_date"]].copy()

movies.head()

,title,overview,genres,vote_average,release_date
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",7.7,1995-10-30
1,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",6.9,1995-12-15
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",6.5,1995-12-22
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",6.1,1995-12-22
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'id': 35, 'name': 'Comedy'}]",5.7,1995-02-10


## 5. Data Cleaning

The dataset contains missing values and inconsistent entries.  
Data cleaning is important because poor data quality can create confusing results and reduce usability.

In this step, I:
- Fill missing text values
- Convert ratings into numeric values
- Remove rows with very short or invalid titles
- Remove rows with very short overviews


In [5]:
movies["title"] = movies["title"].fillna("").astype(str)
movies["overview"] = movies["overview"].fillna("").astype(str)
movies["genres"] = movies["genres"].fillna("").astype(str)
movies["release_date"] = movies["release_date"].fillna("Unknown").astype(str)

movies["vote_average"] = pd.to_numeric(movies["vote_average"], errors="coerce").fillna(0)

movies = movies[movies["title"].str.len() > 1]
movies = movies[movies["overview"].str.len() > 20]

movies.head()

,title,overview,genres,vote_average,release_date
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",7.7,1995-10-30
1,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",6.9,1995-12-15
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",6.5,1995-12-22
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",6.1,1995-12-22
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'id': 35, 'name': 'Comedy'}]",5.7,1995-02-10


## 6. Genre Processing

The genre data is stored as structured text.  
For example, one movie can contain multiple genres such as Action, Adventure, or Science Fiction.

To make the genres usable, I convert the genre text into a Python list.


In [6]:
def extract_genres(text):
    try:
        genres = ast.literal_eval(text)
        if isinstance(genres, list):
            return [genre["name"] for genre in genres if "name" in genre]
        return []
    except:
        return []

movies["genre_list"] = movies["genres"].apply(extract_genres)

movies[["title", "genre_list", "vote_average"]].head()

,title,genre_list,vote_average
0,Toy Story,"[Animation, Comedy, Family]",7.7
1,Jumanji,"[Adventure, Fantasy, Family]",6.9
2,Grumpier Old Men,"[Romance, Comedy]",6.5
3,Waiting to Exhale,"[Comedy, Drama, Romance]",6.1
4,Father of the Bride Part II,[Comedy],5.7


## 7. Final Data Filtering

After extracting the genres, I remove rows without valid genre information.  
I also remove duplicate movie titles.

This improves the reliability of the recommendation output.


In [7]:
movies = movies[movies["genre_list"].map(len) > 0]
movies = movies.drop_duplicates(subset="title")
movies = movies.reset_index(drop=True)

movies[["title", "genre_list", "vote_average", "release_date"]].head()

,title,genre_list,vote_average,release_date
0,Toy Story,"[Animation, Comedy, Family]",7.7,1995-10-30
1,Jumanji,"[Adventure, Fantasy, Family]",6.9,1995-12-15
2,Grumpier Old Men,"[Romance, Comedy]",6.5,1995-12-22
3,Waiting to Exhale,"[Comedy, Drama, Romance]",6.1,1995-12-22
4,Father of the Bride Part II,[Comedy],5.7,1995-02-10


## 8. Iteration Process

The project was developed through several iterations.

### Iteration 1
The first version used raw movie data.  
The system worked, but the movie selection contained noisy and unclear titles, which made the user experience poor.

### Iteration 2
The second version worked technically, but the interface was still difficult to navigate because users had to choose from a long and unclear list of movie titles.

### Iteration 3
The final version switched to genre-based recommendations.  
This made the system easier to use, more stable, and more intuitive.

This process showed that user experience is as important as technical functionality.


## 9. Recommendation Logic

The final recommendation system is based on genre and minimum rating.

The function below:
1. Filters movies by selected genre
2. Filters movies by minimum rating
3. Sorts the results by rating
4. Returns the top recommended movies


In [8]:
def recommend_by_genre(dataframe, genre, min_rating=5.0, n=10):
    filtered = dataframe[
        dataframe["genre_list"].apply(lambda genres: genre in genres)
    ].copy()

    filtered = filtered[filtered["vote_average"] >= min_rating]

    filtered = filtered.sort_values(
        by=["vote_average", "release_date"],
        ascending=[False, False]
    )

    return filtered[["title", "genre_list", "vote_average", "release_date", "overview"]].head(n)

## 10. Example Recommendation

Below is an example output for the genre **Action** with a minimum rating of **6.0**.

This demonstrates how the recommendation logic works before it is used in the Streamlit application.


In [9]:
recommend_by_genre(movies, "Action", min_rating=6.0, n=5)

,title,genre_list,vote_average,release_date,overview
36594,Patient Zero,"[Action, Drama, Horror, Thriller]",10.0,Unknown,After an unprecedented global pandemic has tur...
38589,Tokyo Ghoul,"[Action, Drama, Horror, Thriller]",10.0,2017-07-29,Ken Kaneki (Masataka Kubota) is a university s...
39019,First Round Down,"[Action, Comedy]",10.0,2017-03-04,Tim Tucker (Dylan Bruce) was a star forward wh...
8956,High Roller: The Stu Ungar Story,"[Drama, Action]",10.0,2003-05-01,Based on the true story of the rise and fall o...
27135,Backyard Dogs,"[Action, Comedy]",10.0,2001-11-20,Two teenage boys aspire to win a backyard wres...


## 11. Available Genres

This cell shows the available genres in the cleaned dataset.  
These are the genres that can be used in the recommendation system.


In [10]:
available_genres = sorted({genre for genre_list in movies["genre_list"] for genre in genre_list})

available_genres

['Action',
 'Adventure',
 'Animation',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Family',
 'Fantasy',
 'Foreign',
 'History',
 'Horror',
 'Music',
 'Mystery',
 'Romance',
 'Science Fiction',
 'TV Movie',
 'Thriller',
 'War',
 'Western']

## 12. Reflection

This project showed me that data quality is critical when building data-driven products.

At first, I focused on making the recommendation logic work. However, I later realized that a technically working system is not enough if the user experience is confusing.

The biggest learning point was that simpler solutions can sometimes be more effective than complex ones. By switching from movie-title selection to genre-based recommendation, the app became more user-friendly and stable.

Overall, this was an enjoyable learning process because I could experiment, test different ideas, and improve the final MVP step by step.


## 13. Conclusion

The final result is a simple and functional movie recommendation system.

The project demonstrates:
- Data loading and cleaning
- Genre processing
- Recommendation logic
- Iterative MVP development
- User-focused improvement

The final Streamlit application builds on this notebook and turns the recommendation logic into an interactive data product.
